# Data Cleaning
This script cleans up the Google Exploration data, including filtering out potential bot/spam traffic. There are a lot of comments and test outputs as this was also done as a step for a practicum project.

In [1]:
import pandas as pd
import re # Used for regex

In [2]:
# Load csv data file of Google Analytics Exploration exports from GitHub account
df_ga_exploration = pd.read_csv('https://raw.githubusercontent.com/joedag32/DSSA-5810/refs/heads/main/data/google_exploration_july_10_data.csv')

In [3]:
# Output summary of dataframe to check the datatypes
df_ga_exploration.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7263 entries, 0 to 7262
Data columns (total 50 columns):
 #   Column                                                   Non-Null Count  Dtype 
---  ------                                                   --------------  ----- 
 0   proxy_session_id                                         7263 non-null   object
 1   /graduate/physical-therapy.html                          7263 non-null   int64 
 2   /sciences-math/marine-science.html                       7263 non-null   int64 
 3   /business/hospitality-tourism-program.html               7263 non-null   int64 
 4   /graduate/public-health.html                             7263 non-null   int64 
 5   /arts-humanities/languages-culture.html                  7263 non-null   int64 
 6   /graduate/doctor_nursing_practice.html                   7263 non-null   int64 
 7   /arts-humanities/literature.html                         7263 non-null   int64 
 8   /graduate/education.html              

In [4]:
# Output first 10 rows to get an initial glance
df_ga_exploration.head(10)

,proxy_session_id,/graduate/physical-therapy.html,/sciences-math/marine-science.html,/business/hospitality-tourism-program.html,/graduate/public-health.html,/arts-humanities/languages-culture.html,/graduate/doctor_nursing_practice.html,/arts-humanities/literature.html,/graduate/education.html,/graduate/environmental-science.html,...,/business/business-studies-program.html,/sciences-math/chemistry.html,/social-behavioral-sciences/sociology-anthropology.html,/health-sciences/public-health.html,/social-behavioral-sciences/psychology.html,/sciences-math/biology.html,/social-behavioral-sciences/political-science.html,/sciences-math/biochemistry.html,/arts-humanities/philosophy-religion.html,/business/business-analytics.html
0,/?UPAY_SITE_ID=1_0.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,/?UPAY_SITE_ID=1_11.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,/?UPAY_SITE_ID=1_14.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,/?UPAY_SITE_ID=1_15.0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
4,/?UPAY_SITE_ID=1_16.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,/?fbclid=IwY2xjawS1SjFleHRuA2FlbQIxMABicmlkETJ...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,/?fbclid=IwZXh0bgNhZW0BMABhZGlkAAAv1mvBF9VzcnR...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,1
7,/?fbclid=IwZXh0bgNhZW0BMABhZGlkAAAvwuIe8b1zcnR...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,/?fbclid=PAZXh0bgNhZW0BMABhZGlkAAAvwuHYsV1zcnR...,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
9,/?from=edurank.org_14.0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


I'm happy with my datatypes and any regularization, The Total User metric value is an integer as I was expecting it to be.

In [5]:
# Check how many NaN or Null values are in a variable and output the total
missing_data = df_ga_exploration.isnull().sum()
print(f"Number of NaN or Null values in each variable: \n{missing_data}")

Number of NaN or Null values in each variable: 
proxy_session_id                                           0
/graduate/physical-therapy.html                            0
/sciences-math/marine-science.html                         0
/business/hospitality-tourism-program.html                 0
/graduate/public-health.html                               0
/arts-humanities/languages-culture.html                    0
/graduate/doctor_nursing_practice.html                     0
/arts-humanities/literature.html                           0
/graduate/education.html                                   0
/graduate/environmental-science.html                       0
/health-sciences/health-science.html                       0
/graduate/mba-healthcare-administration.html               0
/health-sciences/exercise-science.html                     0
/business/finance.html                                     0
/sciences-math/geology.html                                0
/graduate/holocaust-genocide-studies.

In [6]:
# Lets see the max value in the variables
max_values = df_ga_exploration.max()
print(f"Max values in each variable: \n{max_values}")

Max values in each variable: 
proxy_session_id                                           /wlfr/news-and-events.html_10.0
/graduate/physical-therapy.html                                                        232
/sciences-math/marine-science.html                                                      34
/business/hospitality-tourism-program.html                                              25
/graduate/public-health.html                                                            30
/arts-humanities/languages-culture.html                                                 20
/graduate/doctor_nursing_practice.html                                                  17
/arts-humanities/literature.html                                                        15
/graduate/education.html                                                               200
/graduate/environmental-science.html                                                   140
/health-sciences/health-science.html                        

Initially I was a bit surprised to see values in the double or higher digits. The metric we're tracking here is Total Users in Google Analytics 4, and a users entire session really should only equate to 1 in most cases. So it's possible that there could be a couple of users with the same session starting around the same time and from the same landing page, but seeing some larger numbers really threw me off. I gave it some thought and research, and conclude it's likely to be spambots, crawlers, or just not humans.

I'm going to rsearch a few of the variables unique values next and see just how varied it is.

In [7]:
# Output unique values in the /graduate/holocaust-genocide-studies.html variable
df_ga_exploration['/graduate/holocaust-genocide-studies.html'].unique()

array([  0,   1,   5,  54,  44,  33,  49,  37,  27,  24,  16,  11,   4,
        15,  14,  17,   3,   9,  30,  32,   2, 127, 118, 529, 516, 510,
       447, 503, 463, 401, 325, 213, 154, 107, 178, 161, 134, 111, 122,
       121, 181, 249, 371, 474,   6,  19,  34,  23,  18,  13,  10,  12,
         7])

In [8]:
# Output unique values in the /graduate/american-studies.html variable
df_ga_exploration['/graduate/american-studies.html'].unique()

array([  0,   1,   2,  12,   8,   7,   5,  13,   3,   4,   6,  49,  40,
        63,  74,  95,  68,  78, 116, 125, 152, 184, 162,  34, 180, 198,
       136, 149,  10,  36,  45,  43,  65,  76])

In [9]:
# Output unique values in the /business/business-analytics.html variable
df_ga_exploration['/business/business-analytics.html'].unique()

array([ 0,  1,  5,  3, 16, 14, 10, 23, 13, 11,  6,  4,  9,  2, 15, 18,  7,
        8])

Digging in a bit deeper, I really suspect we're dealing with some spambots or other web crawlers in our data. Researching further to see how much of it is taking place and then I'll judge if its better to drop or to set to a reasonable session value.

In [10]:
# Number of times 1 value appears in /graduate/holocaust-genocide-studies.html
df_ga_exploration[df_ga_exploration['/graduate/holocaust-genocide-studies.html'] == 1].shape[0]


313

In [11]:
# Number of times a value great than 3 appears in /graduate/holocaust-genocide-studies.html
df_ga_exploration[df_ga_exploration['/graduate/holocaust-genocide-studies.html'] > 3].shape[0]


62

In [12]:
# Number of times 1 value appears in /graduate/american-studies.html
df_ga_exploration[df_ga_exploration['/graduate/american-studies.html'] == 1].shape[0]

222

In [13]:
# Number of times a value great than 3 appears in /graduate/american-studies.html
df_ga_exploration[df_ga_exploration['/graduate/american-studies.html'] > 3].shape[0]

45

In [14]:
# Number of times 1 value appears in /business/business-analytics.html
df_ga_exploration[df_ga_exploration['/business/business-analytics.html'] == 1].shape[0]

138

In [15]:
# Number of times a value great than 3 appears in /business/business-analytics.html
df_ga_exploration[df_ga_exploration['/business/business-analytics.html'] > 3].shape[0]

36

Yeah, the more I dig in the more I suspect spambot or crawlers are giving us those high values and not our metric of Total Users. Can't say I'm surprised given my experience on the web and with Google Analtyics. I'm going to drop any values greater than 3, as they are very likely not actual people visiting the site. A value of 3 could be 2 or 3 legit visitors entering through the same Landing Page at very similar times.

In [16]:
# Check how many rows have values greater than 3
# Create a df without the proxy_session_id column
columns_to_check = df_ga_exploration.columns.drop('proxy_session_id')
# Check is there is a value great than 3 and output the total amount
spam_rows = (df_ga_exploration[columns_to_check] > 3).any(axis=1).sum()
print(f"Number of rows with values greater than 3: {spam_rows}")

Number of rows with values greater than 3: 1287


In [17]:
# Drop any rows with a value greater than 3, they're not real people visiting
df_ga_exploration_filtered = df_ga_exploration[~(df_ga_exploration[columns_to_check] > 3).any(axis=1)].copy()
display(df_ga_exploration_filtered.info())

<class 'pandas.core.frame.DataFrame'>
Index: 5976 entries, 0 to 7262
Data columns (total 50 columns):
 #   Column                                                   Non-Null Count  Dtype 
---  ------                                                   --------------  ----- 
 0   proxy_session_id                                         5976 non-null   object
 1   /graduate/physical-therapy.html                          5976 non-null   int64 
 2   /sciences-math/marine-science.html                       5976 non-null   int64 
 3   /business/hospitality-tourism-program.html               5976 non-null   int64 
 4   /graduate/public-health.html                             5976 non-null   int64 
 5   /arts-humanities/languages-culture.html                  5976 non-null   int64 
 6   /graduate/doctor_nursing_practice.html                   5976 non-null   int64 
 7   /arts-humanities/literature.html                         5976 non-null   int64 
 8   /graduate/education.html                   

None

In [18]:
# Lets see if there are any duplicate rows
dup_rows = df_ga_exploration_filtered.duplicated().sum()
print(f"Number of duplicate rows: {dup_rows}")

Number of duplicate rows: 0


No duplicate rows. I was pretty sure of it, but good to test.

One thing I haven't tested out was the proxy_session_id values. They all should be Landing page + _ + hour. Landing page can be a very unique string, but I can test to make sure a _digit exists in there.

Flagging near duplicate rows was a bit more challenging for me to think of in this case considering the data, but I figured to look into the proxy_session_id column and see what Landing Page urls are in there, and see if they match up popular traffic on the website and that spelling is consistent (though it would be hard for it to not be in this case).

In [19]:
# Function to seperate the Landing Page url from the undercore and hour
def extract_base_url(session_id):
    if isinstance(session_id, str):
        # Use regex to find and remove the last _ followed by digits and optionally a dot and more digits
        match = re.search(r'(_\d+\.?\d*)$', session_id)
        if match:
            return session_id[:match.start()]
        # If no _digit.digit or _digit at the end, return the original string
        return session_id
    return session_id

#Create a just_url variable and assign it just the Landing Page url, no hour or underscore
df_ga_exploration_filtered['just_url'] = df_ga_exploration_filtered['proxy_session_id'].apply(extract_base_url)

# Identify near-duplicate rows based on just_url value
near_duplicate_base_urls = df_ga_exploration_filtered['just_url'].value_counts()
# A row could be considered a near-duplicate if its just_url appears more than once
near_duplicate_base_urls = near_duplicate_base_urls[near_duplicate_base_urls > 1]

print(f"Number of unique Landing Page urls: {len(near_duplicate_base_urls)}")
print("Top 25 Landing Page URLs with the most near-duplicate entries:")
print(display(near_duplicate_base_urls.head(25)))

Number of unique Landing Page urls: 819
Top 25 Landing Page URLs with the most near-duplicate entries:


,count
just_url,
/admissions/index.html,41
/academics/,35
/academic-affairs/academic-schools-programs.html?undergraduate=,34
/academics/index.html,32
/admissions/,32
/admissions/costs.html,31
/admissions/transfer-students.html,30
/graduate/occupational-therapy.html,29
/graduate/,28


None


I was walking the dog and thought that it will probably be good to know how many unique pages were viewd per proxy_session_id as I run this through some models. So I'm going to get that value and assign it to a new variable.

In [20]:
# Identify columns to check for 'unique_views'
# Exclude 'proxy_session_id' and 'just_url' from the count
columns_for_unique_views = df_ga_exploration_filtered.columns.drop(['proxy_session_id', 'just_url'])

# Calculate 'unique_views' by counting how many page view columns have a value > 0 for each session
df_ga_exploration_filtered['unique_views'] = (df_ga_exploration_filtered[columns_for_unique_views] > 0).sum(axis=1)

# Print the unique values in the unique_views variable
print(f"unique_views values: {df_ga_exploration_filtered['unique_views'].unique()}")

unique_views values: [ 1  2  6  4  3  5  8  7 10 11  9 16 14 13 23 15 25 20 19 22 17 21 26 12
 38]


Interesting. There's a total of 39 degree pages, and I highly doubt someone viewed them all in a session. I'm going to set the cutoff at 25, as I suspect anything above that is very likely to not be a human visitor.

In [21]:
# Drop rows where 'unique_views' is greater than 25
df_ga_exploration_filtered = df_ga_exploration_filtered[df_ga_exploration_filtered['unique_views'] <= 25]

# Display the info of the filtered DataFrame to see the new row count
print("DataFrame info after filtering 'unique_views':")
display(df_ga_exploration_filtered.info())

# Display the unique values of 'unique_views' again to confirm the cutoff
print(f"Unique 'unique_views' values after filtering: {df_ga_exploration_filtered['unique_views'].unique()}")

DataFrame info after filtering 'unique_views':
<class 'pandas.core.frame.DataFrame'>
Index: 5974 entries, 0 to 7262
Data columns (total 52 columns):
 #   Column                                                   Non-Null Count  Dtype 
---  ------                                                   --------------  ----- 
 0   proxy_session_id                                         5974 non-null   object
 1   /graduate/physical-therapy.html                          5974 non-null   int64 
 2   /sciences-math/marine-science.html                       5974 non-null   int64 
 3   /business/hospitality-tourism-program.html               5974 non-null   int64 
 4   /graduate/public-health.html                             5974 non-null   int64 
 5   /arts-humanities/languages-culture.html                  5974 non-null   int64 
 6   /graduate/doctor_nursing_practice.html                   5974 non-null   int64 
 7   /arts-humanities/literature.html                         5974 non-null   int64 
 

None

Unique 'unique_views' values after filtering: [ 1  2  6  4  3  5  8  7 10 11  9 16 14 13 23 15 25 20 19 22 17 21 12]


I think the data is in a good spot. Might tweak a few filters like the unique_views after running through models a bit more to make sure its actual visitors and not bots.

In [ ]:
# Export to a CSV
df_ga_exploration_filtered.to_csv('google_exploration_july_10_data_clean.csv', index=False)